# 06 — AI Assistant for Diamond Analysis**Step 6 of the brief.** Takes a question in plain English, finds matching diamonds, explainsthe result like a human expert.## The architecture, and why it is built this wayThe obvious approach — hand the question to an LLM, have it write pandas code, `exec()` it —is a **remote code execution hole** pointed at your own data. Don't build that.This assistant separates the job into three stages:| Stage | Who does it | What it touches ||---|---|---|| **1. Understand** | LLM | Language only. Emits a JSON filter. Never sees the data. || **2. Retrieve** | Our pandas code | The data. Every column and operator checked against a whitelist. || **3. Explain** | LLM | Only the numbers *we* computed. Cannot invent a figure. |**The LLM sits at the edges — language in, language out. The middle is ordinary, inspectable,deterministic Python.** That is the pattern you want any time a language model is anywhere nearreal data, and it is worth far more than this one assignment.It also solves hallucination almost for free. In stage 3 the model is handed a dict of facts wecomputed and asked to *narrate* them. It is never asked to produce a number, so it has noopening to invent one.**Runs without an OpenAI key** — a keyword parser covers stage 1 and a template covers stage 3.

In [ ]:
import sys, pathlibsys.path.append(str(pathlib.Path.cwd().parent))   # [so `from src...` works from notebooks/]import numpy as np, pandas as pdimport matplotlib.pyplot as plt, seaborn as snsfrom src.config import *sns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (9, 5)pd.set_option("display.max_columns", 40)

In [ ]:
from src.data_prep import buildfrom src.assistant import DiamondAssistantdf = build(verbose=False)# [Set OPENAI_API_KEY in your environment for the full experience.#  Without it, the assistant silently drops to the offline fallback.]bot = DiamondAssistant(df)print("LLM mode" if bot.client else "Offline fallback mode (no OPENAI_API_KEY found)")

## Ask it things

In [ ]:
bot.ask("What's the average price of a 1 carat diamond?")

In [ ]:
bot.ask("Show me the cheapest Ideal cut diamonds under $2000")

In [ ]:
bot.ask("How many diamonds have VS1 clarity?")

In [ ]:
bot.ask("I have a $5000 budget and want the biggest stone I can get. What should I look for?")

## Inspect the machinery — this is the part worth understanding

In [ ]:
spec = bot.understand("Find me diamonds over 2 carats with excellent clarity")print("STAGE 1 - the structured filter the LLM produced:")print(spec)facts, sample = bot.retrieve(spec)print("\nSTAGE 2 - facts OUR code computed (the LLM never touches these):")for k, v in facts.items():    print(f"  {k:<14} {v}")print("\nMatching rows:")display(sample)

## The security boundary, demonstratedWatch what happens when the filter asks for something outside the whitelist. It doesn't crash,and it doesn't execute — it is **refused**.

In [ ]:
malicious = {    "filters": [        {"column": "price", "op": "<", "value": 1000},          # [legitimate]        {"column": "__import__('os').system", "op": "==", "value": "rm -rf /"},  # [not]        {"column": "price", "op": "EVAL", "value": "anything"},  # [bad operator]    ],    "aggregate": "mean",}print("Before validation:", len(malicious["filters"]), "filters")safe = bot._validate(malicious)print("After validation: ", len(safe["filters"]), "filters ->", safe["filters"])

The junk is dropped and the legitimate filter survives. **This is why stage 2 is our code and notthe model's.** An LLM can be prompt-injected — a user can talk it into emitting almost anything.It cannot talk our whitelist into growing.Note that we validate the *model's own output*, not just user input. That is the mental shift:**treat the LLM as an untrusted component inside your own system**, not as a trusted colleague.

## Wire the trained models in — the assistant can now *predict*, not just look up

In [ ]:
import joblibfrom tensorflow import kerasfrom src.data_prep import REG_FEATURESreg = keras.models.load_model(MODEL_DIR / "price_regressor.keras")sc  = joblib.load(MODEL_DIR / "price_scaler.pkl")def appraise(carat, cut, color, clarity, depth=61.5, table=57.0):    """Price a diamond that is NOT in the dataset, using the notebook-04 regressor."""    # [Reconstruct the engineered features exactly as data_prep does. If you engineer a    #  feature at train time you must engineer it identically at predict time - a mismatch    #  here is one of the most common ways a deployed model quietly goes wrong.]    x = round((carat / 0.0061) ** (1 / 3), 2)    y, z = x, round(x * depth / 100, 2)    vol = x * y * z    row = {        "carat": carat, "log_carat": np.log1p(carat), "depth": depth, "table": table,        "x": x, "y": y, "z": z, "volume": vol, "density": carat / vol, "xy_ratio": 1.0,        "cut_rank": CUT_MAP[cut], "color_rank": COLOR_MAP[color],        "clarity_rank": CLARITY_MAP[clarity],    }    v = np.array([[row[f] for f in REG_FEATURES]], dtype="float32")    return float(np.expm1(reg.predict(sc.transform(v), verbose=0)[0][0]))for spec_ in [(1.0, "Ideal", "G", "VS1"),              (1.0, "Fair", "J", "SI2"),              (2.0, "Ideal", "D", "IF")]:    print(f"{spec_[0]}ct {spec_[1]:<10} {spec_[2]}  {spec_[3]:<5} -> ${appraise(*spec_):>9,.0f}")

Same carat, wildly different prices — the model has clearly learned that the grades matter, andit has learned it *without* ever being handed the price of a comparable stone. That's thedifference between a lookup and a model.This is also where the project stops being an exercise and starts being a **product**: theassistant can now answer *"what should this stone cost?"* for a diamond that has never existed.